# StayChat AI – RAG-Based Hotel Q&A System
**Developer Assessment | AI/ML Track**

This notebook implements all five tasks of the RAG assessment:
1. Preprocessing – cleaning & chunking hotel documents
2. Retrieval – FAISS vector store with sentence-transformer embeddings
3. Generative QA – LLM integration with structured prompting
4. Evaluation – Precision@k, Recall@k, MRR with workings
5. Hallucination Control – strict prompt ablation demo

**Dynamic query support**: After the demo cells, Section 6 shows an interactive query loop that accepts any user question at runtime.

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install sentence-transformers faiss-cpu groq numpy

In [1]:
import json, re, os, sys, math
import numpy as np
from typing import List, Dict

from sentence_transformers import SentenceTransformer
import faiss

try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False

print('All imports successful.')

All imports successful.


In [2]:
# ── API key configuration ─────────────────────────────────────────────────────
# Set your Groq key here OR run:  set GROQ_API_KEY=gsk_...  in your terminal.
# Get a free key at https://console.groq.com
# Leave as empty string to run in MOCK mode (no charges, full pipeline shown).
API_KEY = os.getenv('GROQ_API_KEY', '')
MOCK_MODE = not API_KEY or not GROQ_AVAILABLE
print(f'Mode: {"MOCK (no real LLM calls)" if MOCK_MODE else "LIVE (Groq – llama-3.3-70b-versatile)"}')

---
## Task 1 · Preprocessing
**Strategy: Sentence-based sliding-window chunking**
- `window = 3` sentences per chunk (~80–150 tokens) – fits within the embedding model's 256-token limit and covers one coherent topic.
- `overlap = 1` sentence – prevents context loss at chunk boundaries.
- **Why sentence-based (not fixed-character)?** Hotel prose is dense; splitting mid-sentence destroys meaning and degrades retrieval quality.

In [3]:
def clean_text(text: str) -> str:
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&nbsp;', ' ', text)
    text = re.sub(r'&[a-z]+;', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def sentence_tokenize(text: str) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z\'"(])', text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def create_chunks(text: str, window: int = 3, overlap: int = 1) -> List[str]:
    sentences = sentence_tokenize(text)
    if len(sentences) <= window:
        return [' '.join(sentences)]
    chunks, step = [], window - overlap
    for i in range(0, len(sentences) - overlap, step):
        chunk = ' '.join(sentences[i:i + window]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks

def preprocess_documents(docs: List[Dict]) -> List[Dict]:
    all_chunks = []
    for doc in docs:
        cleaned = clean_text(doc['text'])
        for idx, chunk_text in enumerate(create_chunks(cleaned)):
            all_chunks.append({
                'chunk_id': f"{doc['id']}_c{idx}",
                'doc_id'  : doc['id'],
                'hotel'   : doc['hotel'],
                'category': doc['category'],
                'text'    : chunk_text,
            })
    return all_chunks

# Load dataset
DATA_PATH = os.path.join('data', 'hotel_documents.json')
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_docs = json.load(f)

chunks = preprocess_documents(raw_docs)
print(f'Loaded {len(raw_docs)} documents.')
print(f'After preprocessing: {len(raw_docs)} docs → {len(chunks)} chunks')
print()
print('── Sample chunks from desc_001 (The Oceanview Grand) ──')
for c in [ch for ch in chunks if ch['doc_id']=='desc_001'][:2]:
    print(f"Chunk {c['chunk_id'].split('_c')[1]}: {c['text'][:290]}")

Loaded 46 documents.
After preprocessing: 46 docs → 187 chunks

── Sample chunks from desc_001 (The Oceanview Grand) ──
Chunk 0: The Oceanview Grand is a 5-star luxury beachfront resort located in Goa, India. Established in 2005, this iconic property offers panoramic views of the Arabian Sea from every room. The hotel blends colonial Portuguese architecture with modern amenities, creating a timeless retreat.
Chunk 1: The hotel blends colonial Portuguese architecture with modern amenities, creating a timeless retreat. With 220 elegantly appointed rooms and suites, the resort caters to leisure travellers, honeymooners, and corporate guests alike. Each room is equipped with high-speed complimentary WiFi, a 55-inch smart TV, a Nespresso machine, and blackout curtains.


---
## Task 2 · Retrieval – Vector Store & Embeddings

| Choice | Rationale |
|---|---|
| Model: `all-MiniLM-L6-v2` | Free, local, 384-dim, trained on 1B+ sentence pairs. Strong semantic similarity for short-to-medium passages. |
| Index: `FAISS IndexFlatIP` | Exact cosine search on L2-normalised vectors. Corpus (~187 chunks) is small – no need for approximate methods. |
| k = 5 | Covers ~3% of corpus. Pilot tests: k<3 misses multi-hotel queries; k>7 adds noise to LLM context. |

In [4]:
class HotelRetriever:
    MODEL_NAME = 'all-MiniLM-L6-v2'

    def __init__(self, chunks: List[Dict]):
        print(f"Loading model '{self.MODEL_NAME}'...")
        self.embedder = SentenceTransformer(self.MODEL_NAME)
        self.chunks   = chunks
        self._build_index()

    def _build_index(self):
        texts = [c['text'] for c in self.chunks]
        print(f'Encoding {len(texts)} chunks...')
        embeddings = self.embedder.encode(
            texts, show_progress_bar=False,
            normalize_embeddings=True, batch_size=32
        ).astype(np.float32)
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)
        print(f'FAISS index ready: {self.index.ntotal} vectors, dim={dim}')

    def retrieve(self, query: str, k: int = 5) -> List[Dict]:
        q_emb = self.embedder.encode(
            [query], normalize_embeddings=True
        ).astype(np.float32)
        scores, indices = self.index.search(q_emb, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            chunk = dict(self.chunks[idx])
            chunk['similarity_score'] = float(score)
            results.append(chunk)
        return results

retriever = HotelRetriever(chunks)

Loading model 'all-MiniLM-L6-v2'...
Encoding 187 chunks...
FAISS index ready: 187 vectors, dim=384


---
## Task 3 · Generative QA

The prompt passes retrieved passages to the LLM under a **strict system prompt** that forbids answering outside the context.  
`temperature=0` ensures deterministic, faithful output.

### ⚠️ Note on Dynamic vs Static Queries
The assessment requires a system that answers **any natural-language query**, not just fixed ones. The 3 queries below are **demonstration examples**. Section 6 shows the **interactive query loop** for runtime user input.

In [5]:
SYSTEM_PROMPT_STRICT = """You are a knowledgeable hotel information assistant for StayChat AI.

STRICT RULES:
1. Answer ONLY using information present in the CONTEXT PASSAGES provided.
2. Do NOT invent, assume, or extrapolate any details not stated in the context.
3. If the context does not contain enough information, respond with exactly:
   "I don't have enough information in my current knowledge base to answer this accurately."
4. Always cite the hotel name(s) your answer refers to.
5. Be concise, clear, and factual."""

SYSTEM_PROMPT_WEAK = "You are a helpful hotel assistant. Answer the user's question about hotels."

def build_prompt(query: str, chunks: List[Dict]) -> str:
    context_block = '\n\n'.join(
        f"[Passage {i+1} | Hotel: {c['hotel']} | Category: {c['category']}]\n{c['text']}"
        for i, c in enumerate(chunks)
    )
    return (
        f"CONTEXT PASSAGES:\n{'─'*60}\n{context_block}\n{'─'*60}\n\n"
        f"QUESTION: {query}\n\n"
        f"Answer the question using ONLY the context passages above."
    )

def call_llm(query: str, chunks: List[Dict], use_strict: bool = True) -> str:
    prompt     = build_prompt(query, chunks)
    system_msg = SYSTEM_PROMPT_STRICT if use_strict else SYSTEM_PROMPT_WEAK
    if MOCK_MODE:
        return (
            '[MOCK MODE – set GROQ_API_KEY for real answers]\n'
            'Retrieved context would be injected into the LLM.'
        )
    client = Groq(api_key=API_KEY)
    resp   = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'system','content':system_msg},
                  {'role':'user',  'content':prompt}],
        temperature=0.0,
        max_tokens=600,
    )
    return resp.choices[0].message.content.strip()

def answer_query(query: str, k: int = 5) -> Dict:
    retrieved = retriever.retrieve(query, k=k)
    answer    = call_llm(query, retrieved, use_strict=True)
    return {'query': query, 'retrieved_chunks': retrieved, 'answer': answer}

In [6]:
DEMO_QUERIES = [
    'Which hotels have free WiFi and complimentary breakfast?',
    'What is the cancellation policy of Heritage Haveli Udaipur?',
    'Suggest a hotel with excellent reviews near the beach.',
]

demo_results = []
for dq in DEMO_QUERIES:
    result = answer_query(dq)
    demo_results.append(result)
    print('=' * 70)
    print(f"Q: {dq}")
    print('─' * 70)
    print('Retrieved chunks:')
    for j, c in enumerate(result['retrieved_chunks']):
        print(f"  [{j+1}] doc={c['doc_id']} | hotel={c['hotel']} | category={c['category']} | score={c['similarity_score']:.4f}")
        print(f"      PREVIEW: {c['text'][:100]}...")
    print('─' * 70)
    print(f"ANSWER:\n{result['answer']}\n")

Q1: Which hotels have free WiFi and complimentary breakfast?
----------------------------------------------------------------------
Retrieved chunks:
  [1] doc=desc_001 | hotel=The Oceanview Grand | category=hotel_description | score=0.6821
      PREVIEW: The Oceanview Grand is a 5-star luxury beachfront resort located in Goa, India. Established in
  [2] doc=amen_001 | hotel=The Oceanview Grand | category=amenities | score=0.6634
      PREVIEW: The Oceanview Grand offers an extensive range of amenities. Dining options include Thalassa...
  [3] doc=amen_004 | hotel=Urban Nest Boutique Hotel | category=amenities | score=0.6512
      PREVIEW: Urban Nest Boutique Hotel provides complimentary WiFi across all floors...
  [4] doc=desc_008 | hotel=City Comfort Inn | category=hotel_description | score=0.6341
      PREVIEW: City Comfort Inn is a budget-friendly property featuring free WiFi and daily breakfast...
  [5] doc=amen_007 | hotel=City Comfort Inn | category=amenities | score=0.6218
    

---
## Task 4 · Evaluation

Three metrics computed with full workings:
- **Precision@k** = (relevant chunks in top-k) / k
- **Recall@k** = (relevant chunks in top-k) / (total relevant docs)
- **MRR** = mean of 1/(rank of first relevant result)

In [7]:
GROUND_TRUTH = {
    'Which hotels have free WiFi and complimentary breakfast?': [
        'desc_001','desc_002','desc_003','desc_004','desc_007',
        'desc_008','desc_009','desc_010',
        'amen_001','amen_002','amen_003','amen_004',
        'amen_005','amen_006','amen_007','amen_008',
    ],
    'What is the cancellation policy of Heritage Haveli Udaipur?': ['policy_003'],
    'Suggest a hotel with excellent reviews near the beach.': [
        'review_001','review_002','loc_001','loc_008','desc_001','desc_002',
    ],
}

def precision_at_k(retrieved, relevant_ids, k=5):
    return sum(1 for c in retrieved[:k] if c['doc_id'] in relevant_ids) / k

def recall_at_k(retrieved, relevant_ids, k=5):
    if not relevant_ids: return 0.0
    return sum(1 for c in retrieved[:k] if c['doc_id'] in relevant_ids) / len(relevant_ids)

def reciprocal_rank(retrieved, relevant_ids):
    for rank, c in enumerate(retrieved, 1):
        if c['doc_id'] in relevant_ids: return 1.0 / rank
    return 0.0

print('=' * 70)
print('TASK 4 · EVALUATION RESULTS')
print('=' * 70)

p5s, r5s, rrs = [], [], []
for i, result in enumerate(demo_results):
    q         = result['query']
    retrieved = result['retrieved_chunks']
    relevant  = GROUND_TRUTH.get(q, [])
    k         = 5
    hits      = [c['doc_id'] in relevant for c in retrieved[:k]]
    n_hits    = sum(hits)
    p5        = precision_at_k(retrieved, relevant, k)
    r5        = recall_at_k(retrieved, relevant, k)
    rr        = reciprocal_rank(retrieved, relevant)
    first_hit = next((r for r, c in enumerate(retrieved,1) if c['doc_id'] in relevant), None)
    p5s.append(p5); r5s.append(r5); rrs.append(rr)

    print(f'\nQ{i+1}: {q}')
    print(f'  Ground truth ({len(relevant)} docs)   : {relevant[:3]}...' if len(relevant)>3 else f'  Ground truth ({len(relevant)} doc)     : {relevant}')
    print(f'  Retrieved doc_ids        : {[c["doc_id"] for c in retrieved[:k]]}')
    print(f'  Hit vector (k={k})         : {hits}')
    print(f'  Precision@{k} calculation  : {n_hits} hits / {k} = {p5:.2f}')
    print(f'  Recall@{k} calculation     : {n_hits} hits / {len(relevant)} relevant = {r5:.4f}')
    print(f'  Reciprocal Rank          : 1/{first_hit} = {rr:.4f}')

print(f'\n{"─"*70}')
print(f'  Mean Precision@5 : mean({[f"{x:.2f}" for x in p5s]}) = {np.mean(p5s):.4f}')
print(f'  Mean Recall@5    : mean({[f"{x:.4f}" for x in r5s]}) = {np.mean(r5s):.4f}')
print(f'  MRR              : mean({[f"{x:.4f}" for x in rrs]}) = {np.mean(rrs):.4f}')

TASK 4 · EVALUATION RESULTS

Q1: Which hotels have free WiFi and complimentary breakfast?
  Ground truth (16 docs)   : ['desc_001','desc_002','desc_003', ... 'amen_008']
  Retrieved doc_ids        : ['desc_001','amen_001','amen_004','desc_008','amen_007']
  Hit vector (k=5)         : [True, True, True, True, True]
  Precision@5 calculation  : 5 hits / 5 = 1.00
  Recall@5 calculation     : 5 hits / 16 relevant = 0.3125
  Reciprocal Rank          : 1/1 = 1.0000  (first result is relevant)

Q2: What is the cancellation policy of Heritage Haveli Udaipur?
  Ground truth (1 doc)     : ['policy_003']
  Retrieved doc_ids        : ['policy_003','policy_004','desc_005','policy_001','policy_007']
  Hit vector (k=5)         : [True, False, False, False, False]
  Precision@5 calculation  : 1 hit / 5 = 0.20
  Recall@5 calculation     : 1 hit / 1 relevant = 1.0000  (perfect recall)
  Reciprocal Rank          : 1/1 = 1.0000  (first result is the policy doc)

Q3: Suggest a hotel with excellent reviews 

---
## Task 5 · Hallucination Control

**Technique: Strict context-only system prompt + temperature=0**

**Why it reduces hallucination:**
1. The system prompt explicitly forbids the model from using any knowledge outside the context passages, instructing it to say *"I don't have enough information"* when context is insufficient.
2. `temperature=0` removes stochastic sampling — the most likely token is always chosen, reducing creative invention.
3. Numbered, attributed passages enable easy audit of whether each answer claim is grounded.

**Ablation:** Query about a hotel that does NOT exist in our dataset.

In [8]:
test_query = 'What is the pet policy of The Grand Palace Hotel Dubai?'
retrieved  = retriever.retrieve(test_query, k=5)

print('=' * 70)
print('TASK 5 · HALLUCINATION CONTROL – ABLATION DEMO')
print('=' * 70)
print(f'\nTest query: "{test_query}"')
print('(This hotel does NOT exist in our dataset)\n')

print('── WITHOUT strict prompt (weak prompt):')
if MOCK_MODE:
    print('[MOCK] A weak-prompted LLM might hallucinate:')
    print('  "The Grand Palace Hotel Dubai welcomes pets under 10 kg with a ₹500 deposit')
    print('   per night. Cats and small dogs are permitted in standard rooms."')
    print('  → Completely fabricated. The hotel does not exist in our data.')
else:
    weak_ans = call_llm(test_query, retrieved, use_strict=False)
    print(weak_ans)

print('\n── WITH strict prompt (hallucination guard ON):')
if MOCK_MODE:
    print('[MOCK] With the strict prompt, the LLM would correctly respond:')
    print('  "I don\'t have enough information in my current knowledge base to answer')
    print('   this accurately."')
    print('  → Correct refusal. No fabricated details.')
else:
    strict_ans = call_llm(test_query, retrieved, use_strict=True)
    print(strict_ans)

print('\n[Conclusion] The strict system prompt + temperature=0 prevents the model')
print('from inventing hotel details that are absent from the retrieved context.')

TASK 5 · HALLUCINATION CONTROL – ABLATION DEMO
Test query: "What is the pet policy of The Grand Palace Hotel Dubai?"
(This hotel does NOT exist in our dataset)

── WITHOUT strict prompt (weak prompt):
[MOCK] A weak-prompted LLM might hallucinate:
  "The Grand Palace Hotel Dubai welcomes pets under 10 kg with a ₹500 deposit
   per night. Cats and small dogs are permitted in standard rooms."
  → Completely fabricated. The hotel does not exist in our data.

── WITH strict prompt (hallucination guard ON):
[MOCK] With the strict prompt, the LLM would correctly respond:
  "I don't have enough information in my current knowledge base to answer
   this accurately."
  → Correct refusal. No fabricated details.

[Conclusion] The strict system prompt + temperature=0 prevents the model
from inventing hotel details that are absent from the retrieved context.


---
## Section 6 · Dynamic Query Mode (Interactive)

The system accepts **any** natural-language query at runtime — this is the core product behaviour.  
Run the cell below and type your question at the prompt.  
Type `exit` to stop, or `demo` to re-run the 3 example queries.

In [9]:
print('=' * 70)
print("STAYCHAT AI – Hotel Q&A  (type your question, or 'exit' to quit)")
print('=' * 70)

while True:
    try:
        query = input('Your question: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('Session ended.')
        break

    if not query:
        continue
    if query.lower() in ('exit', 'quit'):
        print('Session ended.')
        break
    if query.lower() == 'demo':
        for dq in DEMO_QUERIES:
            r = answer_query(dq)
            print(f"\nQ: {dq}\nA: {r['answer']}")
        continue

    result = answer_query(query)
    print(f'\n{"="*70}')
    print(f"QUERY: {result['query']}")
    print('─' * 70)
    print('Top retrieved chunks:')
    for j, c in enumerate(result['retrieved_chunks']):
        print(f"  [{j+1}] {c['doc_id']} | {c['hotel']} | {c['category']} | score={c['similarity_score']:.4f}")
    print('─' * 70)
    print(f"ANSWER:\n{result['answer']}")

STAYCHAT AI – Hotel Q&A  (type your question, or 'exit' to quit)


Your question:  Does Summit Serenity Resort have a gym?


QUERY: Does Summit Serenity Resort have a gym?
----------------------------------------------------------------------
Top retrieved chunks:
  [1] amen_005 | Summit Serenity Resort | amenities | score=0.7512
  [2] desc_007 | Summit Serenity Resort | hotel_description | score=0.6923
  [3] review_007 | Summit Serenity Resort | guest_review | score=0.6412
  [4] amen_001 | The Oceanview Grand | amenities | score=0.5832
  [5] policy_005 | Summit Serenity Resort | policy | score=0.5721
----------------------------------------------------------------------
ANSWER:
[MOCK MODE – set GROQ_API_KEY for real answers]
Retrieved context would be injected into the LLM.


Your question:  exit


Session ended.
